# 23. `num_leaves` under the target-encoded representation

**One variable against ledger row 17** (`lgbm_bag08_seed42_te`, CV 0.966782): the
same nested target and frequency encoder, the same 36 features, the same five
folds, the same seed, the same `learning_rate=0.05`, `n_estimators=2000` and the
same bagging fractions. The only thing that changes is `num_leaves`.

## Why this is being run now

Two facts, and the second is the one that matters.

**`NOTES.md` closed tuning on 2026-08-04**, in the learning-rate entry: "Tuning
stops here. Per the workspace rules, tuning ahead of a stable feature set is the
lowest-yield activity available." That was written on the raw 12 features. Target
encoding landed on 2026-08-11 and turned 12 features into 36, and that feature set
is now stable and LB-confirmed under three separate learners. The precondition the
rule names has been met, so the rule no longer closes this.

**`num_leaves` has never appeared in any notebook in this repo.** It has been
LightGBM's default 31 for the whole competition, on both representations, and it
was never searched at any point. The knobs that were searched are `n_estimators`,
`learning_rate` and the bagging fractions. This is the largest untouched knob here,
not a re-tune of something already looked at.

The prior is not just "it was never tried". `13_target_encoding.ipynb` argues that
target encoding paid because a tree "spends a few dozen splits approximating the
same curve" that the encoding hands it directly. Removing a many-splits cost is
exactly the kind of change that moves what capacity is worth, and 31 leaves was
chosen by LightGBM's authors, not by this data.

## The counterweight, stated up front

Ledger row 34 measured a 0.026 single-model gain converting to **+0.000043** in the
24-member stack, because a fitted combiner extracts most of a direction's value from
a weak member. A single-model gain here is very unlikely to move the stack by its
own size. This run is worth doing because the knob is untested and cheap, not
because the stack is expected to move.

## The gate, pre-registered

Stated before the run so it cannot be chosen after seeing the answer. The test is the
one `NOTES.md` settled in "Fold spread is the wrong yardstick for a paired
comparison": **the spread of the per-fold difference and how many folds it wins**,
against the `num_leaves=31` arm trained in this same kernel on the identical encoded
matrices.

| verdict | condition |
|---|---|
| `carry to a second seed` | wins >= 4/5, paired mean > 2 x paired sd, and paired mean >= 0.00045 |
| `parity` | paired-significant but under 0.00045 |
| `null` | anything else |

`0.00045` is the fold spread these target-encoded models produce, which is
`CLAUDE.md`'s first test. **No arm is called an improvement inside this notebook.**
Row 26 cleared 5/5 folds at t=3.72 and was still logged as parity because only one
seed had been run, and that precedent is binding here.

## What runs

1. Data, folds, the leak checklist.
2. The encoder, fingerprinted against `13` so this is one variable and not two.
3. The leak checks, by execution, reproducing `13`'s printed numbers.
4. Bench and determinism at a short budget, then the projection.
5. The sweep: five folds, the encoder built **once per fold** and every arm trained
   on the identical matrices, so the arms differ by `num_leaves` and nothing else.

In [ ]:
# One flag. The sweep always runs top to bottom on Kaggle.
SMOKE = True

SEED = 42
N_INNER = 5
SMOOTH = 10.0

# The knob under test. 31 is LightGBM's default and therefore ledger row 17's
# value, so that arm doubles as this run's reproduction check.
LEAVES = [15, 31, 63, 127]
BASE_LEAVES = 31

# Row 17's budget, held fixed. Compensating a leaf change with a tree count would
# confound capacity with budget and make this two variables.
LR = 0.05
N_EST = 2000
BENCH_EST = 200
PROBE_FOLD = 0

# Row 17 ran on Kaggle at n_jobs=-1. NOTES.md: deterministic=True pins a result for
# a given thread count, not across thread counts, so matching row 17 means matching
# its thread setting, not picking a better one.
N_JOBS = -1

# Ledger row 17: this feature set, these folds, this seed, num_leaves at the default.
BASELINE_NAME = "lgbm_bag08_seed42_te"
BASELINE_CV = 0.966782
EXPECTED_FOLD_SHA = "ec282b0968059676"

# 13_target_encoding.ipynb printed these. The encoder here must reproduce them.
EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

# The pre-registered gate. See the header.
GATE_FLOOR = 0.00045

print(f"SMOKE = {SMOKE}   leaves {LEAVES}   base {BASE_LEAVES}")

## Stage 1. Data, folds, leak checklist

The fold checksum is the only thing standing between an out-of-fold vector that
blends and one that is silently misaligned, so it is checked before anything trains
rather than after.

In [ ]:
import ast
import gc
import hashlib
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold

# Runs here or on Kaggle. Both are found by name rather than by assuming a shape.
KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
SUB = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "submissions"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
TARGET = "addicted_label"
CAT = ["gender", "stress_level", "academic_work_impact"]
COLS = [c for c in train_full.columns if c not in ("id", TARGET)]

# Leak checklist, re-run rather than ticked by inspection. `id` is a contiguous row
# index that separates train from test perfectly, so it is a guaranteed leak if it
# ever reaches the model.
checks = {
    "id is not a feature": "id" not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full["id"]) & set(test["id"])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != "id"],
}
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(checks.values())

# ROW_IDX maps this run's rows back into the saved member vectors.
if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    N_EST, BENCH_EST = 200, 50
    # Measured 2026-08-19 and written up in NOTES.md: at 16,000 rows this machine
    # runs 101x slower at n_jobs=-1 than at n_jobs=1, monotone in the thread count.
    # N_JOBS above is chosen to match row 17 on Kaggle at 691,369 rows, where it is
    # right. A smoke run produces no ledger number, so overriding it here costs
    # nothing and is the difference between two minutes and giving up on the check.
    N_JOBS = 1
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy()
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print()
print(f"rows {len(train):,}   target rate {y.mean():.6f}")
print(f"fold sizes {np.bincount(folds).tolist()}")
print(f"fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
if SMOKE:
    print("SMOKE: subsampled, so the sha is EXPECTED to differ. Not a check.")
else:
    print("fold alignment: VERIFIED" if ALIGNED else
          "fold alignment: MISMATCH - the OOF from this run is not blendable")

X = train[COLS].copy()
X_test = test[COLS].copy()
for c in CAT:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")

## Stage 2. The encoder

Copied from `13_target_encoding.ipynb` so the feature set is row 17's feature set. A
copy is a provenance risk under the notebook layout, so it is checked rather than
asserted: the cell below parses the encoder out of `13`, normalises both versions
through `ast.unparse`, and compares checksums. Expected fingerprint
`0642e41750ef8bab`, the same value rows 26 and 33 recorded.

In [ ]:
def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    """Semantic checksum of the encoder functions inside a block of source."""
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    if len(parts) != len(ENCODER_FNS):
        return None
    return hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))

theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here        : {mine}")
print(f"encoder fingerprint in 13       : {theirs}")
print(f"rows 26 and 33 recorded         : 0642e41750ef8bab")
print("encoder: IDENTICAL to row 17's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - this is NOT one variable")
print()
print(f"{len(COLS)} raw columns -> {len(COLS) * 3} features after encoding")

### The leak checks, by execution

The same three checks `13` ran, on the same encoder, so their numbers are directly
comparable to the ones in `NOTES.md`. Read all three together: the first two must be
about zero, the third must be large. Without the third, an encoder that ignored the
target entirely would pass the first two and look clean.

In [ ]:
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

# 1. A validation row's own target must never reach its own encoding.
y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

# 2. A training row's own target must never reach its own inner encoding. A small
# residual is expected and is not a leak: `prior` is the training-portion mean.
_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

# 3. The encoding MUST move when targets it is allowed to see change.
y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()

## Stage 3. Bench and determinism

LightGBM is not reproducible by default and this repo found that the hard way, so
`deterministic=True` and `force_row_wise=True` are on and the thread count is the one
row 17 ran at. The check trains the same configuration twice inside this one kernel
and requires bit-identical predictions, which is the only clean form of the test.

The projection matters here because the sweep is four arms rather than one, and the
cost of an arm grows with its leaf count.

In [ ]:
def params(leaves, n_est):
    return dict(
        objective="binary", metric="auc", learning_rate=LR, n_estimators=n_est,
        num_leaves=leaves,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        random_state=SEED, n_jobs=N_JOBS, verbose=-1,
        deterministic=True, force_row_wise=True,
    )


def fit_arm(Xtr, ytr, Xva, leaves, n_est, Xte=None):
    m = lgb.LGBMClassifier(**params(leaves, n_est))
    t0 = time.time()
    m.fit(Xtr, ytr)
    secs = time.time() - t0
    p = m.predict_proba(Xva)[:, 1]
    p_te = m.predict_proba(Xte)[:, 1] if Xte is not None else None
    return p, p_te, secs


def hhmm(s):
    return f"{int(s // 60)}m {int(s % 60):02d}s"


LOG = (Path("/kaggle/working") if ON_KAGGLE
       else LOCAL / "artifacts" / "logs") / "23_lgbm_leaves.log"
LOG.parent.mkdir(parents=True, exist_ok=True)


def note(msg):
    """Print, and append to a log file flushed on every write.

    nbconvert writes the notebook only once the whole run finishes, so without this
    there is no way to watch a long run from outside the kernel. Carried over from
    06 and 17, which learned it the hard way.
    """
    print(msg)
    with LOG.open("a", encoding="utf-8") as fh:
        print(f"{time.strftime('%H:%M:%S')}  {msg}", file=fh, flush=True)


note(f"=== run start, SMOKE={SMOKE}, leaves={LEAVES}, n_jobs={N_JOBS} ===")

_t0 = time.time()
tr0 = np.where(folds != PROBE_FOLD)[0]
va0 = np.where(folds == PROBE_FOLD)[0]
Xtr0, Xva0, _ = build(X, y, tr0, va0)
ENC_SECS = time.time() - _t0
print(f"encoder, one fold: {hhmm(ENC_SECS)}   {Xtr0.shape[1]} features")

pa, _, sa = fit_arm(Xtr0, y[tr0], Xva0, BASE_LEAVES, BENCH_EST)
pb, _, _ = fit_arm(Xtr0, y[tr0], Xva0, BASE_LEAVES, BENCH_EST)
delta = float(np.abs(pa - pb).max())
DETERMINISTIC = delta == 0.0
print(f"lightgbm {lgb.__version__}, n_jobs={N_JOBS}")
print(f"{BENCH_EST} trees at {BASE_LEAVES} leaves on fold {PROBE_FOLD}: "
      f"{hhmm(sa)}, AUC {roc_auc_score(y[va0], pa):.6f}")
print(f"determinism, two identical runs, max |diff|: {delta:.3e}  "
      f"{'OK' if DETERMINISTIC else 'NOT REPRODUCIBLE'}")

# Cost grows with the leaf count, so the projection is per arm rather than flat.
per_tree = sa / BENCH_EST
proj = {n: per_tree * N_EST * (n / BASE_LEAVES) for n in LEAVES}
total = 5 * (ENC_SECS + sum(proj.values()))
print()
print(f"projection, {N_EST} trees, five folds, encoder built once per fold:")
for n in LEAVES:
    print(f"  {n:>4} leaves: {hhmm(proj[n] * 5)}")
print(f"  encoder   : {hhmm(ENC_SECS * 5)}")
print(f"  TOTAL     : {hhmm(total)}")
print()
print("Scaling leaf cost linearly is a rough guide, not a measurement. The real")
print("per-arm times are printed by the sweep below.")
note(f"stage 3 done, determinism {'OK' if DETERMINISTIC else 'FAILED'}, "
     f"projected {hhmm(total)}")

del Xtr0, Xva0, pa, pb
gc.collect()

## Stage 4. The sweep

The encoder is built **once per fold** and every arm trains on the identical
matrices. That is not only cheaper than four separate runs, it is stricter: the arms
cannot differ through the encoder's inner split, so `num_leaves` is the only thing
between them.

In [ ]:
oof = {n: np.zeros(len(train)) for n in LEAVES}
test_pred = {n: np.zeros(len(test)) for n in LEAVES}
per_fold = {n: [] for n in LEAVES}
arm_secs = {n: 0.0 for n in LEAVES}

t0 = time.time()
for f in range(5):
    tr = np.where(folds != f)[0]
    va = np.where(folds == f)[0]
    te0 = time.time()
    Xtr, Xva, Xte = build(X, y, tr, va, X_test)
    note(f"fold {f}: encoded in {hhmm(time.time() - te0)}, {Xtr.shape[1]} features")

    for n in LEAVES:
        p, p_te, secs = fit_arm(Xtr, y[tr], Xva, n, N_EST, Xte)
        oof[n][va] = p
        test_pred[n] += p_te / 5
        per_fold[n].append(float(roc_auc_score(y[va], p)))
        arm_secs[n] += secs
        note(f"  fold {f} leaves {n:>4}: {per_fold[n][-1]:.6f}  ({hhmm(secs)})")

    del Xtr, Xva, Xte
    gc.collect()
    done = time.time() - t0
    note(f"fold {f} done, elapsed {hhmm(done)}, "
         f"about {hhmm(done / (f + 1) * (4 - f))} left")

cv = {n: float(np.mean(per_fold[n])) for n in LEAVES}
sd = {n: float(np.std(per_fold[n])) for n in LEAVES}

print()
print(f"{'leaves':>7} {'CV':>10} {'fold sd':>10} {'time':>10}")
for n in LEAVES:
    star = "  <- row 17's value" if n == BASE_LEAVES else ""
    print(f"{n:>7} {cv[n]:>10.6f} {sd[n]:>10.6f} {hhmm(arm_secs[n]):>10}{star}")
note(f"sweep done in {hhmm(time.time() - t0)}, "
     + ", ".join(f"{n}:{cv[n]:.6f}" for n in LEAVES))

### Does this kernel reproduce row 17?

The `num_leaves=31` arm is row 17's configuration exactly, so it has a number it is
supposed to hit. `NOTES.md` records the cross-machine tolerance: a Kaggle kernel
reproduced ledger row 9 at -6.8e-06 and the local thread-count effect is about 3e-05,
so anything inside 1e-04 is expected and anything outside it means the harness is not
what it claims to be.

**If this check fails, nothing else in the notebook is safe to read**, because every
arm shares the harness it tests.

In [ ]:
repro = cv[BASE_LEAVES] - BASELINE_CV
REPRODUCED = abs(repro) < 1e-4

print(f"arm num_leaves={BASE_LEAVES}: {cv[BASE_LEAVES]:.6f}")
print(f"ledger row 17            : {BASELINE_CV:.6f}")
print(f"difference               : {repro:+.2e}   "
      f"{'inside' if REPRODUCED else 'OUTSIDE'} the 1e-04 tolerance")
if SMOKE:
    print()
    print("SMOKE: a subsampled run against a full-data ledger number, so this is")
    print("EXPECTED to be far outside tolerance and is NOT a check here. It becomes")
    print("one when SMOKE is False, which is the only way this notebook is run on")
    print("Kaggle.")

# The saved vector is the same model's OOF, so it should agree per fold as well as
# on the mean. Skipped in smoke mode, where the folds are a different partition.
if not SMOKE:
    base_oof = np.load(locate("te_bag42_oof.npy"))[ROW_IDX]
    bf = np.array([roc_auc_score(y[folds == f], base_oof[folds == f])
                   for f in range(5)])
    mine_f = np.array(per_fold[BASE_LEAVES])
    print()
    print("per fold, this kernel against the saved row 17 vector:")
    for f in range(5):
        print(f"  fold {f}: {mine_f[f]:.6f} vs {bf[f]:.6f}   "
              f"{mine_f[f] - bf[f]:+.2e}")
    print(f"the saved vector reproduces its own ledger number to "
          f"{bf.mean() - BASELINE_CV:+.2e}")

### The paired comparison and the verdict

Paired per fold against the `num_leaves=31` arm from this same kernel, which is the
right baseline: it shares the folds, the encoder, the seed and the machine, so the
per-fold differences carry none of the variation that cancels.

In [ ]:
base = np.array(per_fold[BASE_LEAVES])
rows = []
for n in LEAVES:
    if n == BASE_LEAVES:
        continue
    d = np.array(per_fold[n]) - base
    rows.append((n, d.mean(), d.std(ddof=1), int((d > 0).sum()), d))

print(f"{'leaves':>7} {'paired mean':>13} {'paired sd':>11} {'wins':>7} "
      f"{'mean/sd':>9}")
for n, m, s, w, _ in rows:
    print(f"{n:>7} {m:>+13.6f} {s:>11.6f} {w:>5}/5 "
          f"{(m / s if s else float('nan')):>9.1f}")
print()
for n, m, s, w, d in rows:
    print(f"  {n:>4}: per-fold {np.round(d, 6).tolist()}")

print()
print("For scale, from the ledger:")
print("  row 17, target encoding   +0.003312, paired sd 0.000270, 5/5")
print("  row 26, CatBoost          +0.000132, paired sd 0.000080, 5/5  -> parity")
print("  row 18, imputation        +0.000023, paired sd 0.000032, 4/5  -> null")

blocked = None
if not (LEAK_OK and CLEAN):
    blocked = "a leak check failed, so nothing measured above is safe to act on"
elif not ENCODER_MATCH:
    blocked = "the encoder does not match 13, so this is not one variable against row 17"
elif not DETERMINISTIC:
    blocked = "the configuration is not reproducible, so a difference between arms \
cannot be separated from run-to-run noise"
elif not SMOKE and not ALIGNED:
    blocked = "fold alignment failed, so these vectors are not blendable"
elif not SMOKE and not REPRODUCED:
    blocked = f"the num_leaves={BASE_LEAVES} arm missed row 17 by {repro:+.2e}, so \
the harness is not row 17's harness and no arm can be compared to it"

print()
if blocked:
    print(f"VERDICT: blocked")
    print(f"  {blocked}")
else:
    best = max(rows, key=lambda r: r[1]) if rows else None
    n, m, s, w, _ = best
    sig = w >= 4 and s > 0 and m > 2 * s
    if sig and m >= GATE_FLOOR:
        v = "carry to a second seed"
        why = (f"num_leaves={n} clears the pre-registered bar at {m:+.6f}. Per row "
               f"26's precedent one seed is not enough to log an improvement, so "
               f"the next run is this arm at a second seed and nothing else.")
    elif sig:
        v = "parity"
        why = (f"num_leaves={n} is paired-significant at {m:+.6f} but under the "
               f"{GATE_FLOOR:.5f} floor. Log it as parity, not as an improvement, "
               f"the same treatment row 26 got.")
    else:
        v = "null"
        why = ("no arm is both positive and clear of its own paired noise. The "
               "default of 31 stands, and num_leaves is closed.")
    print(f"VERDICT: {v}")
    print(f"  {why}")

if SMOKE:
    print()
    print("SMOKE: everything above is computed on subsampled rows and is NOT a")
    print("result. Its only purpose is to prove these branches execute. The fold")
    print("spread of a smoke run is several times the real one, so the gate cannot")
    print("resolve anything and its wording must not be quoted.")

In [ ]:
pre = "SMOKE_" if SMOKE else ""
for n in LEAVES:
    np.save(OUT / f"{pre}te_leaves{n}_oof.npy", oof[n])
    np.save(OUT / f"{pre}te_leaves{n}_test.npy", test_pred[n])
print(f"wrote {pre}te_leaves<n>_oof.npy and _test.npy for {LEAVES}")

# No submission csv is written here. Nothing in this notebook has cleared the bar to
# be submitted, and a csv sitting in submissions/ is one `kaggle competitions submit`
# away from spending a slot on a result the gate above did not authorise.
print()
print("ledger lines, one per arm:")
for n in LEAVES:
    print(f"  te_leaves{n:<4}  cv_mean {cv[n]:.6f}  cv_std {sd[n]:.6f}")
print()
print(f"  leak checks {'PASS' if CLEAN else 'FAILED'}, "
      f"fold alignment {'verified' if ALIGNED else 'NOT verified'}, "
      f"encoder {'matches 13' if ENCODER_MATCH else 'DIFFERS'}, "
      f"determinism {'OK' if DETERMINISTIC else 'FAILED'}")
print(f"  num_leaves={BASE_LEAVES} reproduces row 17 to {repro:+.2e}")